# Week 2 – Data Collection, Cleaning and Preprocessing for Logistics Analysis

This notebook implements the YuvaIntern Week 2 preprocessing pipeline using a compact simulated logistics dataset inspired by the publicly available DataCo SMART SUPPLY CHAIN FOR BIG DATA ANALYSIS dataset. The original source data is not redistributed here.


## Dataset reference
Constante, F., Silva, F., & Pereira, A. (2019). DataCo SMART SUPPLY CHAIN FOR BIG DATA ANALYSIS. Mendeley Data, Version 5. DOI: 10.17632/8gx2fvg2k6.5

Reference: https://data.mendeley.com/datasets/8gx2fvg2k6/5


In [ ]:
import pandas as pd
import numpy as np

raw = pd.read_csv('../data/week2_simulated_raw_logistics_data.csv')
print('Shape:', raw.shape)
display(raw.head())


In [ ]:
# Inspect data quality
print('Missing values:')
display(raw.isna().sum().sort_values(ascending=False))
print('Duplicate rows:', raw.duplicated().sum())
display(raw.describe(include='all').T)


In [ ]:
# Standardize categorical values
for col in ['Shipping_Mode','Region','Customer_Segment','Product_Category']:
    raw[col] = raw[col].astype('string').str.strip()

mode_map = {
    'standard class':'Standard Class','STANDARD CLASS':'Standard Class',
    'first class':'First Class','FIRST CLASS':'First Class',
    'second class':'Second Class','SECOND CLASS':'Second Class',
    'same day':'Same Day','SAME DAY':'Same Day'
}
raw['Shipping_Mode'] = raw['Shipping_Mode'].replace(mode_map)
raw['Shipping_Mode'] = raw['Shipping_Mode'].fillna(raw['Shipping_Mode'].mode()[0])

region_map = {'west':'West','WEST':'West','east':'East','EAST':'East',
              'central':'Central','CENTRAL':'Central','south':'South','SOUTH':'South'}
raw['Region'] = raw['Region'].replace(region_map)
raw['Region'] = raw['Region'].fillna(raw['Region'].mode()[0])


In [ ]:
# Dates and numeric missing values
raw['Order_Date'] = pd.to_datetime(raw['Order_Date'], errors='coerce')
raw['Order_Date'] = raw['Order_Date'].fillna(raw['Order_Date'].median())

numeric_cols = ['Shipment_Quantity','Product_Price','Shipping_Cost',
                'Scheduled_Shipping_Days','Actual_Shipping_Days','Customer_Rating']
for col in numeric_cols:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')
    raw[col] = raw[col].fillna(raw[col].median())

print('Remaining missing values:', raw.isna().sum().sum())


In [ ]:
# Remove duplicate rows
before = len(raw)
raw = raw.drop_duplicates().reset_index(drop=True)
print('Rows removed:', before-len(raw))


In [ ]:
# Detect and cap outliers using IQR
for col in ['Product_Price','Shipping_Cost']:
    q1=raw[col].quantile(0.25); q3=raw[col].quantile(0.75)
    iqr=q3-q1
    lower=q1-1.5*iqr; upper=q3+1.5*iqr
    mask=(raw[col]<lower)|(raw[col]>upper)
    print(col, 'outliers:', mask.sum())
    raw[col]=raw[col].clip(lower,upper)


In [ ]:
# Feature engineering
raw['Delay_Days'] = raw['Actual_Shipping_Days'] - raw['Scheduled_Shipping_Days']
raw['Late_Delivery_Flag'] = (raw['Delay_Days'] > 0).astype(int)

# Min-Max normalization
for col in ['Shipment_Quantity','Product_Price','Shipping_Cost','Customer_Rating']:
    mn=raw[col].min(); mx=raw[col].max()
    raw[col+'_Normalized']=(raw[col]-mn)/(mx-mn)

display(raw.head())


In [ ]:
# Final validation and export
print('Final shape:', raw.shape)
print('Remaining missing values:', raw.isna().sum().sum())
print('Remaining duplicates:', raw.duplicated().sum())
raw.to_csv('../data/week2_cleaned_logistics_data.csv', index=False)
print('Cleaned dataset saved.')
